[Énoncés : Série 1 — Listes, chaînes, dictionnaires et ensembles](../serie1_listes_chaines_dicts_sets.ipynb)

# Corrigé — Série 1 — Listes, chaînes, dictionnaires et ensembles

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec ses tests. Il existe souvent plusieurs bonnes solutions : si la tienne passe tous les tests et le test de vitesse, elle est juste. Compare surtout l'**idée**.

Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 1 — Alterner les extrémités

**Idée** : deux indices qui partent des deux bouts et se rapprochent (thème 3). On lit la liste sans la modifier. Piège : quand la taille est impaire, les deux indices finissent sur le même élément, qu'il ne faut ajouter qu'une fois.

**Complexité** : O(n). À éviter : `nombres.pop(0)` dans une boucle, qui coûte O(n) à chaque fois → O(n²).

In [ ]:
def alterner_extremes(nombres: list[int]) -> list[int]:
    resultat = []
    gauche, droite = 0, len(nombres) - 1
    while gauche <= droite:
        resultat.append(nombres[gauche])
        if gauche != droite:                 # taille impaire : l'élément du milieu une seule fois
            resultat.append(nombres[droite])
        gauche += 1
        droite -= 1
    return resultat


verifier(alterner_extremes, [
    (([10, 20, 30, 40, 50],), [10, 50, 20, 40, 30]),
    (([1, 2, 3, 4],), [1, 4, 2, 3]),
    (([],), []),   # liste vide
    (([7],), [7]),
    (([1, 2],), [1, 2]),
    (([5, 5, 1],), [5, 1, 5]),
], ne_pas_modifier=True)

In [ ]:
# Test de vitesse
chrono(alterner_extremes, list(range(100_000)))

## Exercice 2 — Premier caractère unique

**Idée** : c'est `premier_caractere_unique` du thème 1, mais on renvoie l'indice. Un dict compteur (passage 1), puis un second passage **dans l'ordre du texte** pour trouver le premier caractère de compte 1.

**Complexité** : O(n). À éviter : `texte.count(c)` pour chaque caractère, qui est une boucle cachée → O(n²).

In [ ]:
def indice_premier_unique(texte: str) -> int:
    compte = {}
    for c in texte:                          # passage 1 : compter chaque caractère
        compte[c] = compte.get(c, 0) + 1
    for i, c in enumerate(texte):            # passage 2 : relire dans l'ordre du texte
        if compte[c] == 1:
            return i
    return -1


verifier(indice_premier_unique, [
    (('aabbcdde',), 4),
    (('aabb',), -1),
    (('',), -1),   # chaîne vide
    (('x',), 0),
    (('aA',), 0),
    (('abcabd',), 2),
    (('zzyzx',), 2),
])

In [ ]:
# Test de vitesse
chrono(indice_premier_unique, "ab" * 40_000 + "c", attendu=80_000)

## Exercice 3 — Compresser les groupes de lettres

**Idée** : `i` marque le début d'un groupe, `j` avance tant que le caractère est le même ; la longueur du groupe est `j - i`, puis on repart de `j`. Chaque caractère est lu une seule fois.

**Complexité** : O(n). On accumule les morceaux dans une liste et on fait un seul `"".join(...)` à la fin.

In [ ]:
def compresser_groupes(texte: str) -> str:
    morceaux = []
    i = 0
    while i < len(texte):
        j = i
        while j < len(texte) and texte[j] == texte[i]:   # j avance jusqu'à la fin du groupe
            j += 1
        morceaux.append(texte[i] + str(j - i))           # caractère + longueur du groupe
        i = j                                            # le groupe suivant commence en j
    return "".join(morceaux)


verifier(compresser_groupes, [
    (('aaabbc',), 'a3b2c1'),
    (('aabaa',), 'a2b1a2'),
    (('',), ''),   # chaîne vide
    (('z',), 'z1'),
    (('aaaaaaaaaaaa',), 'a12'),   # groupe de plus de 9 : deux chiffres
    (('aA',), 'a1A1'),
])

## Exercice 4 — Badges d'entrée et de sortie

**Idée** : un **set** des badges actifs. `add` pour une entrée, `discard` pour une sortie : contrairement à `remove`, `discard` ne lève pas d'erreur si le nom est absent, ce qui gère directement « sortie ignorée ». On trie une seule fois, à la fin.

**Complexité** : O(n + a log a), avec a le nombre de badges actifs à la fin.

In [ ]:
def badges_actifs(evenements: list[str]) -> list[str]:
    actifs = set()
    for evenement in evenements:
        nom, symbole = evenement[:-1], evenement[-1]   # le symbole est le dernier caractère
        if symbole == "+":
            actifs.add(nom)
        else:
            actifs.discard(nom)        # discard ne plante pas si le nom est absent (≠ remove)
    return sorted(actifs)


verifier(badges_actifs, [
    ((['alice+', 'bob+', 'alice-', 'alice+', 'zoe+'],), ['alice', 'bob', 'zoe']),
    ((['bob-', 'bob+', 'bob+'],), ['bob']),
    ((['anna+', 'anna-'],), []),
    (([],), []),
    ((['zoe+', 'marc+', 'eva+'],), ['eva', 'marc', 'zoe']),
    ((['marc+', 'marc-', 'marc-'],), []),
])

## Exercice 5 — Paires de même parité

**Idée** : inutile de regarder les paires une par une. Il suffit de compter les pairs (`p`) et les impairs (`q`) : parmi `p` éléments, on forme `p × (p − 1) / 2` paires. Variante tout aussi bonne : en lisant la liste, ajouter à chaque élément le nombre d'éléments **déjà vus** de même parité.

**Complexité** : O(n). La double boucle serait en O(n²).

In [ ]:
def paires_meme_parite(nombres: list[int]) -> int:
    pairs = sum(1 for x in nombres if x % 2 == 0)
    impairs = len(nombres) - pairs
    # parmi p éléments, on peut former p × (p − 1) / 2 paires différentes
    return pairs * (pairs - 1) // 2 + impairs * (impairs - 1) // 2


verifier(paires_meme_parite, [
    (([1, 2, 3, 4, 6],), 4),
    (([],), 0),   # liste vide
    (([5],), 0),
    (([2, 4],), 1),
    (([1, 2],), 0),
    (([7, 7, 7, 7],), 6),
    (([-3, 1, 0, 2],), 2),   # négatifs et zéro
])

In [ ]:
# Test de vitesse
chrono(paires_meme_parite, list(range(10_000)), attendu=24_995_000)

## Exercice 6 — Classer les mots les plus fréquents

**Idée** : un dict compteur, puis un tri avec une **clé** qui est un tuple `(-fréquence, mot)`. Python compare les tuples élément par élément : d'abord la fréquence (le signe `-` inverse l'ordre), puis, à égalité, le mot.

**Complexité** : O(n + d log d), avec d le nombre de mots distincts.

In [ ]:
def classement_frequences(mots: list[str]) -> list[str]:
    compte = {}
    for mot in mots:
        compte[mot] = compte.get(mot, 0) + 1
    # clé de tri : fréquence décroissante (d'où le signe −), puis ordre alphabétique
    return sorted(compte, key=lambda mot: (-compte[mot], mot))


verifier(classement_frequences, [
    ((['chat', 'chien', 'chat', 'oiseau', 'chien', 'chat'],), ['chat', 'chien', 'oiseau']),
    ((['b', 'a', 'c'],), ['a', 'b', 'c']),   # toutes à égalité : ordre alphabétique
    (([],), []),
    ((['pomme', 'kiwi', 'pomme', 'kiwi', 'banane'],), ['kiwi', 'pomme', 'banane']),
    ((['x'],), ['x']),
])

---

[Énoncés de la série](../serie1_listes_chaines_dicts_sets.ipynb)